# Demo: unusual cases & clash-safety of “by construction” methods

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/Demo_Unusual_Cases_Clash_Safety.ipynb)

**Notebook ID:** `Demo_Unusual_Cases_Clash_Safety` — *not* the 5-minute survey reproduce notebook.

Demonstrates how ChiralFold handles:
1. **Cyclic / strained peptides** (1XT7 daptomycin)
2. **Non-standard CCD ligands** (1OF6 DTY←L-Tyr mislabels)
3. **Ultra-high-resolution errors** (1HHZ @ 0.99 Å — not a density artifact)
4. **Non-protein Rama exclusion** (5M2K vancomycin glycopeptide)
5. **Clashscore invariance** under L↔D mirror (isometry)
6. **AF3 correction** preserves CA–N/C/Cβ bonds and drives residual violations to 0

Related (different notebook): [`Reproduce_PDB_D_Residue_Errors_5min.ipynb`](Reproduce_PDB_D_Residue_Errors_5min.ipynb)

In [ ]:
import os
if not os.path.isdir('chiralfold'):
    !git clone --depth 1 https://github.com/Tommaso-R-Marena/ChiralFold.git
    %cd ChiralFold
!pip -q install "chiralfold==3.5.1"
import chiralfold
print('ChiralFold', chiralfold.__version__)

In [ ]:
# --- Unusual case table from frozen artefacts ---
import json, pandas as pd
from IPython.display import display, Markdown

cases = [
    {'PDB': '1XT7', 'Class': 'Macrocycle / cyclic lipopeptide', 'Issue': 'DSG labeled D but L coords (Stereochem)', 'Res': 'NMR'},
    {'PDB': '1OF6', 'Class': 'Non-standard CCD ligand', 'Issue': '8× DTY should be TYR (CCD-Code)', 'Res': '2.1 Å'},
    {'PDB': '1HHZ', 'Class': 'Ultra-high resolution', 'Issue': 'DAL error at 0.99 Å — not low-density artifact', 'Res': '0.99 Å'},
    {'PDB': '5M2K', 'Class': 'Glycopeptide (not protein)', 'Issue': 'Excluded from Rama benchmark (pre-specified)', 'Res': '1.0 Å'},
]
display(pd.DataFrame(cases))

excl = json.load(open('results/5m2k_benchmark_exclusion.json'))
display(Markdown(f"""### 5M2K exclusion
**{excl.get('title', excl.get('pdb_id'))}** — {excl.get('benchmark_exclusion_reason', '')[:280]}
"""))

In [ ]:
# --- Clashscore: mirror isometry on toy ubiquitin fragment ---
from chiralfold import audit_pdb, mirror_pdb, correct_af3_output, detect_chirality_violations
import tempfile, os

src = 'chiralfold/data/examples/toy_ubiquitin_fragment.pdb'
b = audit_pdb(src)
out = tempfile.mktemp(suffix='_m.pdb')
mirror_pdb(src, out, axis='x', rename_residues=True)
a = audit_pdb(out)
print('Mirror clashscore before/after:', b['clashes']['clash_score'], '→', a['clashes']['clash_score'])
print('n_clashes before/after:', b['clashes']['n_clashes'], '→', a['clashes']['n_clashes'])
assert b['clashes']['n_clashes'] == a['clashes']['n_clashes']
print('PASS — mirror does not invent steric clashes.')
os.remove(out)

In [ ]:
# --- AF3 correction on synthetic inverted Ala ---
inv = 'chiralfold/data/examples/synthetic_l_ala_inverted.pdb'
before = detect_chirality_violations(inv)
out = tempfile.mktemp(suffix='_c.pdb')
result = correct_af3_output(inv, out)
after = result['after']
print('Violations before:', before['n_violations'], 'after:', after['n_violations'])
assert after['n_violations'] == 0
# Clashscore on tiny 1-residue systems is often 0 both sides
ba = audit_pdb(inv)['clashes']
aa = audit_pdb(out)['clashes']
print('Clashscore before/after correction:', ba['clash_score'], '→', aa['clash_score'])
print('PASS — residual chirality violations = 0 after correction.')
os.remove(out)

In [ ]:
# --- Audit a cyclic peptide error structure if cached locally ---
from pathlib import Path
for pdb_id in ['1XT7', '1OF6', '1HHZ']:
    p = Path(f'results/d_survey/{pdb_id}.pdb')
    if p.is_file():
        r = audit_pdb(str(p))
        print(f"{pdb_id}: score={r['overall_score']:.1f}  chirality_wrong={r['chirality']['n_wrong']}  clashes={r['clashes']['n_clashes']}")
    else:
        print(f'{pdb_id}: not in local cache (see CSV survey)')